---
authors:
  - edesz
date: 2025-10-04
---

# Get Best Features and Model from Validation

## About

In this step, we will evaluate all 24 runs across the five ML experiments performed as part of the validation analysis performed in the [previous step](./04_run_validation_exeriments.ipynb).

So, here, we will determine the best combination of inputs to the ML pipeline: features, feature preprocessing and classifier (ML model).

:::{note}
### Outputs

None
:::

## Python Imports

The required Python modules are imported below

In [ ]:
import pandas as pd
from great_tables import GT, loc, md, style
from metaflow import Flow, metadata

Import a custom Python module for post-processing the outputs of cross-validation

In [ ]:
import cc_churn.tuning as tn

## User Inputs

Below we define variables that will be used later in this notebook

In [ ]:
primary_metric_val = "prauc"
threshold_overfit = 5

cols_expt_compare = [
    "experiment_num",
    "run_id",
    "feat_group",
    "model_name",
    "threshold",
    "fit_time",
    "score_time",
    f"train_{primary_metric_val}",
    f"test_{primary_metric_val}",
    "pct_diff",
    "is_overfit",
    "is_overfit_significant",
]

## Comparison of Validation Experiments

The results produced by the experiments to optimize **feature selection** and **model selection** are now compared.

Get a list of all Metaflow flow runs

In [ ]:
metaflow_experiment_runs = list(Flow("ValidationFlow").runs())

Verify all runs are finished

In [ ]:
assert all(list(map(lambda x: x.finished, metaflow_experiment_runs)))

Results from all experiments are combined and sorted in descending order of the primary evaluation metric for model validation (`prauc`) 

In [ ]:
cv_results_tuned_models = {
    f"{run.data.df_cv['model_name'].head(1).squeeze()}_{run.id}": (
        run.data.df_cv.assign(
            run_id=run.id,
            experiment_num=lambda df: (
                df["feat_group"].map(
                    {
                        "[numericals_1]": 1,
                        "[numericals_2]": 2,
                        "[numericals_1,ordinals]": 3,
                        "[numericals_1,ordinals,categoricals_ohe_encoding]": 4,
                        "[numericals_1,ordinals,categoricals_no_encoding]": 5,
                    }
                )
            ),
        )
    )
    for k, run in enumerate(metaflow_experiment_runs, 1)
}
df_cv_scores_outer = tn.combine_cv_scores_thresholds(cv_results_tuned_models)
df_cv_scores_all = tn.agg_cv_scores_thresholds(
    df_cv_scores_outer, primary_metric_val, threshold_overfit
)

### Numericals (Two Groupings)

Results from all six runs of experiments 1 and 2, which compared the choice of two groupings of numerical features, are shown below

In [ ]:
df_cv_scores_expts_1_2 = (
    df_cv_scores_all[cols_expt_compare]
    .query("experiment_num.isin([1,2])")
    .sort_values(by=["model_name", "experiment_num"], ignore_index=True)
)

This is shown below

In [ ]:
gt = (
    GT(df_cv_scores_expts_1_2.drop(columns=["feat_group"]))
    .tab_style(style=style.text(weight="bold"), locations=loc.column_header())
    .tab_header(
        md(
            "**Scores from Experiments to Choose Best Group of Numerical "
            "Features**"
        )
    )
    .tab_style(
        style=[style.fill(color="teal"), style.text(color="white")],
        locations=loc.body(columns=["model_name"]),
    )
    .tab_style(
        style=style.fill(color="yellow"),
        locations=loc.body(columns=[f"test_{primary_metric_val}"]),
    )
    .tab_style(
        style=[
            style.fill(color="red"),
            style.text(color="white", weight="bold"),
        ],
        locations=loc.body(columns=["is_overfit_significant"]),
    )
    .fmt_number(
        columns=[
            "threshold",
            "fit_time",
            "score_time",
            "train_prauc",
            "test_prauc",
            "pct_diff",
        ],
        decimals=3,
    )
)
gt

:::{attention}
Model performance is assessed using the primary scoring metric for model validation discussed in the project scope, namely `prauc`. Here, we are using this metric calculated on the data that was not used in model training, so the column with the score to consider is `test_prauc`.
:::

**Observations**

1. The first grouping of numerical features outperforms the second grouping for the tree-based models, but not for the linear models. The model training time was nearly unchanged across both feature groupings.
2. The two best performing classifiers were `XGBClassifier` (second best) and `HistGradientBoostingClassifier` (best).

### Numericals, Numericals+Ordinals, Numericals+Ordinals+OHE Categoricals

Results from all runs of the following experiments are shown below

1. experiment 1 - only using numerical features
2. experiment 3 - using numerical and ordinal features
3. experiment 4 - using all features, including categoricals with one-hot encoding

In [ ]:
df_cv_scores_expts_1_3_4 = (
    df_cv_scores_all[cols_expt_compare]
    .query("experiment_num.isin([1,3,4])")
    .sort_values(by=["model_name", "experiment_num"], ignore_index=True)
)

This is shown below

In [ ]:
gt = (
    GT(df_cv_scores_expts_1_3_4.drop(columns=["feat_group"]))
    .tab_style(style=style.text(weight="bold"), locations=loc.column_header())
    .tab_header(
        md(
            "**Scores from Experiments to Test Hypothesis that Ordinal (3) and "
            "Categorical (4) Features are not Predictive of Churn**"
        )
    )
    .tab_style(
        style=[style.fill(color="teal"), style.text(color="white")],
        locations=loc.body(columns=["model_name"]),
    )
    .tab_style(
        style=style.fill(color="yellow"),
        locations=loc.body(columns=[f"test_{primary_metric_val}"]),
    )
    .tab_style(
        style=[
            style.fill(color="red"),
            style.text(color="white", weight="bold"),
        ],
        locations=loc.body(columns=["is_overfit_significant"]),
    )
    .fmt_number(
        columns=[
            "threshold",
            "fit_time",
            "score_time",
            "train_prauc",
            "test_prauc",
            "pct_diff",
        ],
        decimals=3,
    )
)
gt

**Observations**

1. 1. For `XGBClassifier`, performance with the ordinal features fell by ~0.1%. With categorical features, it improved by ~0.1%.
2. For `RandomForestClassifier`, `LogisticRegression_imbalanced`, `Ensemble__VotingClassifier` and `LogisticRegression`, performance with the ordinal features improved by ~0.1%. With categorical features, it fell by ~0.1%.
3. Except for `RandomForestClassifier` and `Ensemble__VotingClassifier`, the inclusion of these two groups of features did consistently increase the training time (see `fit_time`).
4. (ensembles-mixed-strength-learners)=
   For each expeiment run, the `Ensemble__VotingClassifier` has a lower score than one of the two best performing tree-based classifiers used in the same run (`XGBClassifier` or `HistGradientBoostingClassifier`) since it also included weaker models such as `LogisticRegression` and `RandomForestClassifier`.

### Numericals, Numericals+Ordinals, Numericals+Ordinals+Unencoded Categoricals

Results from all runs of the following experiments are shown below

1. experiment 1 - only using numerical features
2. experiment 3 - using numerical and ordinal features
3. experiment 5 - using all features, including categoricals with categorical encoding performed by the classifier

In [ ]:
df_cv_scores_expts_1_3_5 = (
    df_cv_scores_all[cols_expt_compare]
    .query("experiment_num.isin([1,3,5])")
    .query("model_name == 'HistGradientBoostingClassifier'")
    .sort_values(by=["model_name", "experiment_num"], ignore_index=True)
)

This is shown below

In [ ]:
gt = (
    GT(df_cv_scores_expts_1_3_5.drop(columns=["feat_group"]))
    .tab_style(style=style.text(weight="bold"), locations=loc.column_header())
    .tab_header(
        md(
            "**Scores from Experiments to Compare Numerical (1), Ordinal (3) and "
            "Unencoded Categorical (5) Features**"
        )
    )
    .tab_style(
        style=[style.fill(color="teal"), style.text(color="white")],
        locations=loc.body(columns=["model_name"]),
    )
    .tab_style(
        style=style.fill(color="yellow"),
        locations=loc.body(columns=[f"test_{primary_metric_val}"]),
    )
    .tab_style(
        style=[
            style.fill(color="red"),
            style.text(color="white", weight="bold"),
        ],
        locations=loc.body(columns=["is_overfit_significant"]),
    )
    .fmt_number(
        columns=[
            "threshold",
            "fit_time",
            "score_time",
            "train_prauc",
            "test_prauc",
            "pct_diff",
        ],
        decimals=3,
    )
)
gt

**Observations**

1. The inclusion of ordinal and/or unencoded categorical features (experiments 3 and 5) did not improve pipeline performance (`test_prauc`) relative to performance using numerical features only (experiment 1).

### General Observations

1. In all experiments in which linear models were used, tree-based models doubled the performance of the linear models.
2. In all the experiments in which it was used, `HistGradientBoostingClassifier` and `XGBClassifier` and nearly identical performance (`test_prauc`) and training time (`fit_time`). Both outperformed `Ensemble__VotingClassifier`, and strongly outperformed `RandomForestClassifier`. The added disadvantage of `Ensemble__VotingClassifier` is that its training times take longer than other two.
3. In all experiments with top-performing tree-based models (`Ensemble__VotingClassifier`, `XGBClassifier`, `HistGradientBoostingClassifier`), overfitting (in the `pct_diff` column) falls in the 5-10% range. `XGBClassifier` consistently had higher overfitting than `HistGradientBoostingClassifier`. For linear models and `RandomForestClassifier`, overfitting is less than 2%.
4. As seen from comparing experiments 1 or 2 (which did not include sensitive features) to 3, 4 or 5 (which did include sensitive features), model performance was not negatively impacted if sensitive features were excluded. This verifies the hypothesis we formed in the [EDA step](./03_eda_v2.ipynb) that only numerical features are sufficient to predict credit card customer churn in the sample of customer data used for this project.

### Main Findings

:::{important}
Based on the combination of the primary evaluation metric, training time and overfitting across all runs of the five experiments, the best combination of pipeline steps (features, pre-processor and ML classifier) is to use only the first grouping of numerical features that are min-max scaled (in experiment 1) and the `HistGradientBoostingClassifier` classifier model.

:::{attention}
All sensitive features can be excluded from the training data without negatively impacting model performance on customer data in this dataset. This eliminates the possibliity of discrimination related to these attributes.
:::

(best-validation-outputs)=
### Get Metadata for Best Experiment

Based on the mean outer CV scores, get the best

1. experiment run (`run_id`)
2. experiment number
3. classifier name
4. average decision threshold

In [ ]:
# get best experiment run
df_cv_scores_best = df_cv_scores_all.query("(experiment_num == 1)").nlargest(
    1, [f"test_{primary_metric_val}"]
)

# get best Metaflow run ID
best_experiment_run_id = df_cv_scores_best["run_id"].squeeze()

# get best experiment number
best_experiment_num = df_cv_scores_best["experiment_num"].squeeze()

# get best classifier
best_model_name = df_cv_scores_best["model_name"].squeeze()

# get best decision threshold
best_estimator_threshold = df_cv_scores_best["threshold"].squeeze()
# print(best_experiment_run_id)

#### Estimate of Model Reliability

The standard deviation of the prediction scores across outer CV folds indicates a model's stability within each experiment. A high standard deviation suggests the model is sensitive to specific training data subsets, potentially indicating high variance or overfitting.

We will estimate model reliability from the outer CV scores during model validation by calculating the [coefficient of variation (COV)](https://en.wikipedia.org/wiki/Coefficient_of_variation), which is the ratio of standard deviation to the mean, of the primary validation metric for all experiment runs

In [ ]:
df_cov = (
    pd.concat(
        [
            # get mean and standard deviation of train and test fold scores
            (
                df_cv.groupby(
                    ["feat_group", "experiment_num", "run_id", "model_name"]
                )
                .agg(
                    {
                        f"train_{primary_metric_val}": ["mean", "std"],
                        f"test_{primary_metric_val}": ["mean", "std"],
                    }
                )
                .set_axis(
                    [
                        f"train_mean_{primary_metric_val}",
                        f"train_std_{primary_metric_val}",
                        f"test_mean_{primary_metric_val}",
                        f"test_std_{primary_metric_val}",
                    ],
                    axis=1,
                )
                .reset_index()
            )
            # iterate over all CV folds for all experiment runs
            for k, df_cv in cv_results_tuned_models.items()
        ]
    )
    # sort by experiment number
    .sort_values(by=["experiment_num"], ascending=True, ignore_index=True)
    # calculate train COV and test fold COV
    .assign(
        train_cov=lambda df: (
            df[f"train_std_{primary_metric_val}"]
            .div(df[f"train_mean_{primary_metric_val}"])
            .mul(100)
        ),
        test_cov=lambda df: (
            df[f"test_std_{primary_metric_val}"]
            .div(df[f"test_mean_{primary_metric_val}"])
            .mul(100)
        ),
    )
)

This is shown below

In [ ]:
gt = (
    GT(df_cov.drop(columns=["feat_group"]))
    .tab_style(style=style.text(weight="bold"), locations=loc.column_header())
    .tab_header(
        md(
            f"**Coefficient of Variation of {primary_metric_val} in Train and "
            "Test Fold Scores Across All CV Folds**"
        )
    )
    .tab_style(
        style=[style.fill(color="teal"), style.text(color="white")],
        locations=loc.body(columns=["model_name"]),
    )
    .tab_style(
        style=style.fill(color="yellow"),
        locations=loc.body(columns=["run_id"]),
    )
    .tab_style(
        style=[
            style.fill(color="red"),
            style.text(color="white", weight="bold"),
        ],
        locations=loc.body(columns=["train_cov", "test_cov"]),
    )
    .fmt_number(
        columns=[
            f"train_mean_{primary_metric_val}",
            f"train_std_{primary_metric_val}",
            f"test_mean_{primary_metric_val}",
            f"test_std_{primary_metric_val}",
            "train_cov",
            "test_cov",
        ],
        decimals=3,
    )
)
gt

**Observations**

1. Across all outer CV folds for all classifiers (models) and across all experiment runs, the COV of the primary validation metric (`prauc`) is at most 6.5% on the validation split of each CV fold. This is higher than the COV of the training split per CV fold of at most 4.3%. This is partially explained by the relatively smaller size of the validation fold (~80:20 split per CV fold). For most runs, the COV is lower for the training split than for the validation split.
2. For the best classifier (`HistGradientBoostingClassifier`), the COV is consistently between 1%-4%. It is reassuring that COVs are relatively small for this ML model (less than ~4.1%).
3. For `XGBClassifier`, adding ordinal (3) or catgegorical (4) features reduces the COV but, on average, it is still higher on the test split than on the train split. For `HistGradientBoostingClassifier`, adding ordinal or categorical features reduces the test COV to less than the train COV.

This suggests the larger COV on unseen (validation) data is primarily due to its relatively smaller size or choice of features and/or classifier and that this does not indicate an underlying difference in the unseen data relative to the training data.

## Limitations

1. Hyperparameter optimization was not performed.
2. Not all models (classifiers) that support un-encoded categorical features were compared (eg. `XGBClassifier`).
3. [Ensembles contain weak and strong classifiers](#ensembles-mixed-strength-learners). Future work should exclude the weak learners.
4. [Right-skewed histograms of numerical features during EDA](./03_eda_v2.ipynb) suggest linear models would benefit from log-scaling. Future work should explore the addition of a pre-procesing step that performs log scaling.

## Conclusion

The inclusion of ordinal or categorical features made a negligible impact on model performance using `prauc`. This validates the hypothesis we formed during EDA that numerical features were sufficient to predict credit card customer churn.

Tree-based models have outperformed linear models. In the EDA notebook, we saw the need for using tree-based models from the skewed disributions of the numerical features that were used in model training. This finding about the relative outperformance by tree-based models verifies that observation.

In both types of models, overfitting is present but negligible.

Based on model performance, training time and overfitting, the validation analysis here indicates

1. the best features are numerical features only
2. features should be preprocessed using `scikit-learn`'s `MinMaxScaler()`
3. the best model (classifier) is `scikit-learn`'s `HistGradientBoostingClassifier`